# Lecture: Clinical Research Informatics - Sommer Semester 2026
Exercise Sheet: 5

Additional information:   
used additional packages: pandas 2.3.3   
dataset used: Coding Data 02 - standard.zip

In [16]:
import os.path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

# Functions

In [17]:
def filter_by_patientlist(df, pat_ids):
    return df[df["PATIENT"].isin(pat_ids)]


def calc_current_age(birthday):
    today = pd.to_datetime('now')
    age = pd.to_timedelta(today - birthday)
    age = age.dt.days / 365.2425
    return np.round(age)


def get_age_of_condition_start(df_pat: pd.DataFrame, df_cond: pd.DataFrame):
    temp = pd.merge(df_cond, df_pat, how="left", left_on="PATIENT", right_on="Id")
    age = pd.to_timedelta(temp["START"] - temp["BIRTHDATE"])
    age = age.dt.days / 365.2425
    return np.round(age)


def filter_by_age_range(df_pat: pd.DataFrame, min_age: int, max_age: int):
    return df_pat[(df_pat['CURRENT_AGE'] >= min_age) & (df_pat['CURRENT_AGE'] <= max_age)]


def get_encounter_numbers_per_patient(df_enc, code):
    return df_enc[df_enc["CODE"] == code]["PATIENT"].value_counts()


# Section 1: Data Exploration and Preparation

### 1.1 Import datasets and show dimensions

In [18]:
DATA_FILES = {
    "patients": "patients.csv",
    "conditions": "conditions_reduced.csv",
    "careplans": "careplans.csv",
    "allergies": "allergies.csv",
    "medications": "medications_reduced.csv",
    "encounters": "encounter_reduced.csv",
    "observations": "observations_reduced.csv",
}

date_columns = {
    "patients": ["BIRTHDATE", "DEATHDATE"],
    "conditions": ["START", "STOP"],
    "careplans": ["START", "STOP"],
    "allergies": ["START", "STOP"],
    "medications": ["START", "STOP"],
    "encounters": ["START", "STOP"],
    "observations": ["DATE"],
}

dataframes = {}
for name, filename in DATA_FILES.items():
    df = pd.read_csv(os.path.join("data/2_standard", filename), parse_dates=date_columns[name])
    dataframes[name] = df
    print("{}: {} variables (columns), {} entries (rows)".format(filename, df.shape[1], df.shape[0]))

df_pat = dataframes["patients"]
df_cond = dataframes["conditions"]
df_enc = dataframes["encounters"]
df_med = dataframes["medications"]
df_obs = dataframes["observations"]
df_cp = dataframes["careplans"]
df_all = dataframes["allergies"]

patients.csv: 27 variables (columns), 58307 entries (rows)
conditions_reduced.csv: 6 variables (columns), 481427 entries (rows)
careplans.csv: 9 variables (columns), 204096 entries (rows)
allergies.csv: 15 variables (columns), 51846 entries (rows)
medications_reduced.csv: 13 variables (columns), 131382 entries (rows)
encounter_reduced.csv: 15 variables (columns), 78274 entries (rows)
observations_reduced.csv: 9 variables (columns), 1048575 entries (rows)


### 1.2 Add age columns and print averages

In [19]:
df_pat["CURRENT_AGE"] = calc_current_age(df_pat["BIRTHDATE"])
df_cond["START_AGE"] = get_age_of_condition_start(df_pat, df_cond)
print(f"Average age of the patients is {df_pat['CURRENT_AGE'].mean():.4f}.")
print(f"Average age of condition start is {df_cond['START_AGE'].mean():.4f}.")

Average age of the patients is 47.2773.
Average age of condition start is 46.3577.


### 1.3 Filter patients aged 6–12 years

In [20]:
print(f"Patients table has {len(df_pat.index)} rows before filtering.")
df_pat = filter_by_age_range(df_pat, 6, 12)
print(f"Patients table has {len(df_pat.index)} rows after filtering.")

Patients table has 58307 rows before filtering.
Patients table has 3927 rows after filtering.


# 1.4	Create a method that allows to filter tables with a given list of patient IDs. Use this method to filter
for children who are diagnosed with childhood asthma. Print the number of records before and after your filtering.
How many children have more than one condition? What is the maximum number of conditions?

In [ ]:
asthma_patients = df_cond[df_cond["CODE"].isin([233678006])]["PATIENT"]  # filter for childhood asthma
df_pat = df_pat[df_pat["Id"].isin(asthma_patients)]

print(f"Patients table has {len(df_pat.index)} rows after filtering for Asthma.")
print(f"Average age of the patients is {df_pat['CURRENT_AGE'].mean():.0f}.")
print(f"Conditions table has {len(df_cond.index)} rows before filtering.")
print(f"Encounter table has {len(df_enc.index)} rows before filtering.")
print(f"Allergies table has {len(df_all.index)} rows before filtering.")
print(f"Medications table has {len(df_med.index)} rows before filtering.")
print(f"Observations table has {len(df_obs.index)} rows before filtering.")
print(f"Careplans table has {len(df_cp.index)} rows before filtering.")

df_cond = filter_by_patientlist(df_cond, df_pat["Id"])
df_enc = filter_by_patientlist(df_enc, df_pat["Id"])
df_all = filter_by_patientlist(df_all, df_pat["Id"])
df_med = filter_by_patientlist(df_med, df_pat["Id"])
df_obs = filter_by_patientlist(df_obs, df_pat["Id"])
df_cp = filter_by_patientlist(df_cp, df_pat["Id"])

print(f"Conditions table has {len(df_cond.index)} rows after filtering.")
print(f"Encounter table has {len(df_enc.index)} rows after filtering.")
print(f"Allergies table has {len(df_all.index)} rows after filtering.")
print(f"Medications table has {len(df_med.index)} rows after filtering.")
print(f"Observations table has {len(df_obs.index)} rows after filtering.")  
print(f"Careplans table has {len(df_cp.index)} rows after filtering.")


Patients table has 300 rows after filtering for Asthma.
Average age of the patients is 9.
Conditions table has 481427 rows before filtering.
Encounter table has 78274 rows before filtering.
Conditions table has 1400 rows after filtering.
Encounter table has 9010 rows after filtering.


In [22]:
condition_counts = df_cond["PATIENT"].value_counts()
num_children_multimorbid = (condition_counts > 1).sum()
print(f"{num_children_multimorbid} children have more than one condition.")
print(f"The maximum number of conditions for a child is {condition_counts.max()}.")

289 children have more than one condition.
The maximum number of conditions for a child is 11.


# 2.1	Consider ways to quantify the health status of these asthmatic children. Extract relevant information from the available datasets and add them to the patient DataFrame.


In [23]:
encounter_counts = df_enc["PATIENT"].value_counts()
print(f"Average number of encounter: {encounter_counts.mean():.2f}")
print(f"Maximum number of encounter: {encounter_counts.max()}")

encounter_frequencies = df_enc["DESCRIPTION"].value_counts()
condition_frequencies = df_cond["DESCRIPTION"].value_counts()

df_pat = df_pat.drop_duplicates()
df_pat.index = df_pat["Id"]

df_pat["TOTAL_CONDITIONS"] = condition_counts
df_pat["TOTAL_ENCOUNTER"] = encounter_counts
df_pat["ASTHMA_REASON_ENCOUNTER"] = df_enc[df_enc["REASONCODE"] == 233678006]["PATIENT"].value_counts()  # encounter due to asthma
df_pat["SYMPTOM_ENCOUNTER"] = get_encounter_numbers_per_patient(df_enc, 185345009)  # Encounter for symptom
df_pat["EMERGENCY_ENCOUNTER"] = get_encounter_numbers_per_patient(df_enc, 50849002)  # emergency hospital admission
df_pat["INPATIENT_ENCOUNTER"] = get_encounter_numbers_per_patient(df_enc, 305408004)  # inpatient encounter
df_pat["URGENT_CARE_ENCOUNTER"] = get_encounter_numbers_per_patient(df_enc, 702927004)  # urgent care clinic
df_pat["ASTHMA_FU_ENCOUNTER"] = get_encounter_numbers_per_patient(df_enc, 394701000)  # asthma follow-up
df_pat["ASTHMA_MED_REQUEST"] = df_med[df_med["REASONCODE"] == 233678006]["PATIENT"].value_counts() # medication request for asthma

Average number of encounter: 30.03
Maximum number of encounter: 421


In [24]:
# Active allergy flag
df_pat["HAS_ACTIVE_ALLERGY"] = (
    df_all[df_all["STOP"].isna()]
    .groupby("PATIENT")["CODE"].count()
    .gt(0).astype(int)
)

In [25]:
# Active careplan flag
df_pat["HAS_ACTIVE_CAREPLAN"] = (
    df_cp[df_cp["STOP"].isna()]
    .groupby("PATIENT")["CODE"].count()
    .gt(0).astype(int)
)

In [32]:
df_cp.isna().sum()

Id                        0
START                     0
STOP                 105927
PATIENT                   0
ENCOUNTER                 0
CODE                      0
DESCRIPTION               0
REASONCODE            95471
REASONDESCRIPTION     95471
dtype: int64

# 3.1	Export your working data into a csv file.

In [27]:
export_path = os.path.join("data", "working_data")
os.makedirs(export_path, exist_ok=True)
df_pat.to_csv(os.path.join(export_path, "week_5_data.csv"), index=None)